In [1]:
import pandas as pd
import os
from pathlib import Path

## Load data

### Download VEF embeddings & parquet files

In [2]:
DOWNLOAD_DIR = "/mnt/localdisk/data/embeddings/vef/panda/"

In [ ]:
!mkdir -p $DOWNLOAD_DIR
!azcopy copy "https://kaiko.blob.core.windows.net/long-term-experimental/embeddings/panda/data/embeddings/experiments/pathology_fm/tcga/20240209/" $DOWNLOAD_DIR --overwrite=prompt --check-md5 FailIfDifferent --from-to=BlobLocal --recursive --log-level=INFO;

In [15]:
!azcopy copy "https://kaiko.blob.core.windows.net/long-term-experimental/embeddings/panda/splits/stratified_split_noisy_labels_removed.parquet" $DOWNLOAD_DIR;
!azcopy copy "https://kaiko.blob.core.windows.net/long-term-experimental/embeddings/panda/labels/multiclass_classification.parquet" $DOWNLOAD_DIR;

INFO: Scanning...
INFO: Autologin not specified.
INFO: Authenticating to source using Azure AD
INFO: Any empty folders will not be processed, because source and/or destination doesn't have full folder support

Job 2cc7c74a-98e7-e64f-6148-b3808c19688c has started
Log file is located at: /Users/nkaenzig/.azcopy/2cc7c74a-98e7-e64f-6148-b3808c19688c.log

INFO: azcopy 10.21.0: A newer version 10.23.0 is available to download

100.0 %, 1 Done, 0 Failed, 0 Pending, 0 Skipped, 1 Total, 2-sec Throughput (Mb/s): 1.4883


Job 2cc7c74a-98e7-e64f-6148-b3808c19688c summary
Elapsed Time (Minutes): 0.0334
Number of File Transfers: 1
Number of Folder Property Transfers: 0
Number of Symlink Transfers: 0
Total Number of Transfers: 1
Number of File Transfers Completed: 1
Number of Folder Transfers Completed: 0
Number of File Transfers Failed: 0
Number of Folder Transfers Failed: 0
Number of File Transfers Skipped: 0
Number of Folder Transfers Skipped: 0
TotalBytesTransferred: 372295
Final Job Status: Comp

## Load & merge VEF parquet & old manifest file

In [3]:
old_manifest_path = os.path.join(DOWNLOAD_DIR, "20240209/dino_vits16/version_0/checkpoints/last/predictions.csv")

df_old_manifest = pd.read_csv(old_manifest_path)

df_old_manifest.head()

,filename,save_path,metadata,pt_index
0,04198f362b0329843b23c5cd236f765d,az://long-term-experimental@kaiko.blob.core.wi...,"{""x"": ""2032"", ""y"": ""10922"", ""level"": ""0""}",0
1,04198f362b0329843b23c5cd236f765d,az://long-term-experimental@kaiko.blob.core.wi...,"{""x"": ""2286"", ""y"": ""10414"", ""level"": ""0""}",1
2,04198f362b0329843b23c5cd236f765d,az://long-term-experimental@kaiko.blob.core.wi...,"{""x"": ""2540"", ""y"": ""9652"", ""level"": ""0""}",2
3,04198f362b0329843b23c5cd236f765d,az://long-term-experimental@kaiko.blob.core.wi...,"{""x"": ""2540"", ""y"": ""10922"", ""level"": ""0""}",3
4,04198f362b0329843b23c5cd236f765d,az://long-term-experimental@kaiko.blob.core.wi...,"{""x"": ""2794"", ""y"": ""10922"", ""level"": ""0""}",4


In [4]:
splits_path = os.path.join(DOWNLOAD_DIR, "stratified_split_noisy_labels_removed.parquet")
labels_path = os.path.join(DOWNLOAD_DIR, "multiclass_classification.parquet")

df_splits = pd.read_parquet(splits_path)
df_labels = pd.read_parquet(labels_path)

display(df_splits.head())
display(df_labels.head())

,path,split
0,az://ml-datasets-public@kaiko.blob.core.window...,train
1,az://ml-datasets-public@kaiko.blob.core.window...,train
2,az://ml-datasets-public@kaiko.blob.core.window...,train
3,az://ml-datasets-public@kaiko.blob.core.window...,train
4,az://ml-datasets-public@kaiko.blob.core.window...,train


,path,label
0,az://ml-datasets-public@kaiko.blob.core.window...,0
1,az://ml-datasets-public@kaiko.blob.core.window...,0
2,az://ml-datasets-public@kaiko.blob.core.window...,4
3,az://ml-datasets-public@kaiko.blob.core.window...,4
4,az://ml-datasets-public@kaiko.blob.core.window...,0


In [5]:
df = pd.merge(df_splits, df_labels, on="path", how="inner")
df["slide_id"] = df["path"].apply(lambda x: Path(x).stem)

df.head()

,path,split,label,slide_id
0,az://ml-datasets-public@kaiko.blob.core.window...,train,4,aa9be7d9f82e983d21e2746078b877d9
1,az://ml-datasets-public@kaiko.blob.core.window...,train,1,34a98ca2d4eb1a91e428bf2112e26543
2,az://ml-datasets-public@kaiko.blob.core.window...,train,4,95eeb46ecc4a9693119627fedb8df55c
3,az://ml-datasets-public@kaiko.blob.core.window...,train,1,1df32b02eaa3cfad5d8c51a3e289cfc1
4,az://ml-datasets-public@kaiko.blob.core.window...,train,2,ebb6d5ca45942536f78beb451ee43cc4


In [6]:
select_columns = ["save_path", "split", "label", "slide_id"]

df = pd.merge(df, df_old_manifest, left_on="slide_id", right_on="filename", how="inner")
df = df[select_columns].drop_duplicates().reset_index(drop=True)

df.head()


,save_path,split,label,slide_id
0,az://long-term-experimental@kaiko.blob.core.wi...,train,4,aa9be7d9f82e983d21e2746078b877d9
1,az://long-term-experimental@kaiko.blob.core.wi...,train,4,aa9be7d9f82e983d21e2746078b877d9
2,az://long-term-experimental@kaiko.blob.core.wi...,train,4,aa9be7d9f82e983d21e2746078b877d9
3,az://long-term-experimental@kaiko.blob.core.wi...,train,4,aa9be7d9f82e983d21e2746078b877d9
4,az://long-term-experimental@kaiko.blob.core.wi...,train,4,aa9be7d9f82e983d21e2746078b877d9


In [7]:
abs_prefix = "az://long-term-experimental@kaiko.blob.core.windows.net/embeddings/panda/data/embeddings/experiments/pathology_fm/tcga/"

df["path"] = df["save_path"].str.replace(abs_prefix, "")

df.drop(columns=["save_path"], inplace=True)

df.head()

,split,label,slide_id,path
0,train,4,aa9be7d9f82e983d21e2746078b877d9,20240209/dino_vits16/version_0/checkpoints/las...
1,train,4,aa9be7d9f82e983d21e2746078b877d9,20240209/dino_vits16/version_0/checkpoints/las...
2,train,4,aa9be7d9f82e983d21e2746078b877d9,20240209/dino_vits16/version_0/checkpoints/las...
3,train,4,aa9be7d9f82e983d21e2746078b877d9,20240209/dino_vits16/version_0/checkpoints/las...
4,train,4,aa9be7d9f82e983d21e2746078b877d9,20240209/dino_vits16/version_0/checkpoints/las...


## Save the new manifest

In [8]:
df = df.rename(columns={"label": "target"})

In [28]:
save_path = "/mnt/localdisk/data/embeddings/vef/panda/20240209/dino_vits16/manifest.csv"

df.to_csv(save_path, index=False)

In [9]:
len(df)

51420

In [26]:
p = "/Users/nkaenzig/workspace/eva-worktrees/eva/vef_panda_manifest.csv"

df_vef_manifest = pd.read_csv(p)
df_vef_manifest["path"] = df_vef_manifest["path"].str.replace(abs_prefix, "")

df_vef_manifest.head()

,split,target,slide_id,data_provider,path
0,train,4,aa9be7d9f82e983d21e2746078b877d9,radboud,20240209/dino_vits16/version_0/checkpoints/las...
1,train,4,aa9be7d9f82e983d21e2746078b877d9,radboud,20240209/dino_vits16/version_0/checkpoints/las...
2,train,4,aa9be7d9f82e983d21e2746078b877d9,radboud,20240209/dino_vits16/version_0/checkpoints/las...
3,train,4,aa9be7d9f82e983d21e2746078b877d9,radboud,20240209/dino_vits16/version_0/checkpoints/las...
4,train,4,aa9be7d9f82e983d21e2746078b877d9,radboud,20240209/dino_vits16/version_0/checkpoints/las...


In [28]:
save_path = "/mnt/localdisk/data/embeddings/vef/panda/20240209/dino_vits16/manifest.csv"

df_vef_manifest.reset_index(drop=True).to_csv(save_path, index=False)

In [27]:
len(df_vef_manifest), len(df)

(51420, 51420)

In [25]:
pd.merge(df_vef_manifest, df, on="path", how="inner").shape

(51420, 8)

In [17]:
df_vef_manifest["slide_id"].value_counts()

slide_id
35a2ba270ef1fe91e4434cc83ccac50d    10
a9a2bdda9278e216b19a3a948f4f29ad    10
665fcad8a2159796d111633809f6d0b3    10
fd0bb45eba479a7f7d953f41d574bf9f    10
d2029423ffecf93f8114770e5d061027    10
                                    ..
a420d76844dd359a2d3a268be4827768     4
0e5806abc1cf909123d584e504dd9bf9     4
f29962cf62f852b83823f7ab438f64d7     4
dc7825252bd5c07e9a31c0651c989c77     4
c0b74d23f8502804f3e047ff5129a57b     3
Name: count, Length: 9555, dtype: int64

In [18]:
df["slide_id"].value_counts()

slide_id
35a2ba270ef1fe91e4434cc83ccac50d    10
a9a2bdda9278e216b19a3a948f4f29ad    10
665fcad8a2159796d111633809f6d0b3    10
fd0bb45eba479a7f7d953f41d574bf9f    10
d2029423ffecf93f8114770e5d061027    10
                                    ..
a420d76844dd359a2d3a268be4827768     4
0e5806abc1cf909123d584e504dd9bf9     4
f29962cf62f852b83823f7ab438f64d7     4
dc7825252bd5c07e9a31c0651c989c77     4
c0b74d23f8502804f3e047ff5129a57b     3
Name: count, Length: 9555, dtype: int64